# Bidirectional LSTM over a padded batch

Reproduce what `nn.LSTM(bidirectional=True, batch_first=True)` computes on a padded batch fed through `pack_padded_sequence`, from the raw weights and without `nn.LSTM`, `nn.LSTMCell` or the packing utilities. The hard part is padding: the backward direction must start at each sequence's **last real token**, not at position `T - 1`.

Write `bilstm_padded(x, lengths, W_ih_f, W_hh_f, b_ih_f, b_hh_f, W_ih_b, W_hh_b, b_ih_b, b_hh_b)`:

- `x`: float64 `(B, T, D)`, batch first. Positions `t >= lengths[b]` are padding and may hold any values.
- `lengths`: int64 `(B,)`, each in `1..T`, in any order.
- Per direction (`_f` forward, `_b` backward), in `nn.LSTM`'s layout: `W_ih` `(4H, D)`, `W_hh` `(4H, H)`, `b_ih`, `b_hh` `(4H,)`. The four row blocks are the gates in the order **i, f, g, o**.
- Initial hidden and cell states are zero for both directions.

One step, with $z = x_t W_{ih}^\top + b_{ih} + h_{t-1} W_{hh}^\top + b_{hh}$ split into $z_i, z_f, z_g, z_o$:

$$c_t = \sigma(z_f) \odot c_{t-1} + \sigma(z_i) \odot \tanh(z_g), \qquad h_t = \sigma(z_o) \odot \tanh(c_t)$$

The forward direction reads `t = 0, 1, ..., len - 1`; the backward direction reads `t = len - 1, ..., 0`.

## Return
A tuple `(out, h_n, c_n)`:

- `out`: `(B, T, 2H)`. `out[b, t, :H]` is the forward hidden state after reading position `t`; `out[b, t, H:]` is the backward hidden state after reading position `t`. Both halves are 0 at padded positions.
- `h_n`, `c_n`: `(2, B, H)`. Index 0 holds the forward direction's final state (after position `len - 1`), index 1 the backward direction's final state (after position 0).

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def bilstm_padded(x, lengths, W_ih_f, W_hh_f, b_ih_f, b_hh_f, W_ih_b, W_hh_b, b_ih_b, b_hh_b):
    """Return (out (B, T, 2H), h_n (2, B, H), c_n (2, B, H))."""
    B, T, _ = x.shape
    H = W_hh_f.shape[1]

    def run(W_ih, W_hh, b_ih, b_hh, times):
        h = x.new_zeros(B, H)
        c = x.new_zeros(B, H)
        outs = x.new_zeros(B, T, H)
        for t in times:
            z = x[:, t] @ W_ih.T + b_ih + h @ W_hh.T + b_hh
            i, f, g, o = z.chunk(4, dim=1)
            c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
            h = torch.sigmoid(o) * torch.tanh(c)
            outs[:, t] = h
        return outs, h, c

    # TODO: padding is ignored, so short sequences read garbage and the backward pass starts in the padding
    of, hf, cf = run(W_ih_f, W_hh_f, b_ih_f, b_hh_f, range(T))
    ob, hb, cb = run(W_ih_b, W_hh_b, b_ih_b, b_hh_b, range(T - 1, -1, -1))
    return torch.cat([of, ob], dim=-1), torch.stack([hf, hb]), torch.stack([cf, cb])


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0], [1.0]]], "dtype": "float64"}, {"__nd__": [2], "dtype": "int64"}, {"__nd__": [[0.5], [0.5], [0.5], [0.5]], "dtype": "float64"}, {"__nd__": [[0.5], [0.5], [0.5], [0.5]], "dtype": "float64"}, {"__nd__": [0.5, 0.5, 0.5, 0.5], "dtype": "float64"}, {"__nd__": [0.5, 0.5, 0.5, 0.5], "dtype": "float64"}, {"__nd__": [[-0.5], [-0.5], [-0.5], [-0.5]], "dtype": "float64"}, {"__nd__": [[-0.5], [-0.5], [-0.5], [-0.5]], "dtype": "float64"}, {"__nd__": [-0.5, -0.5, -0.5, -0.5], "dtype": "float64"}, {"__nd__": [-0.5, -0.5, -0.5, -0.5], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.5143859225346401, -0.03593879277619472], [0.7612810361872527, -0.029851680305425975]]], "dtype": "float64"}, {"__nd__": [[[0.7612810361872527]], [[-0.03593879277619472]]], "dtype": "float64"}, {"__nd__": [[[1.434689906675743]], [[-0.19713365441048528]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0], [1.0], [1.0]], [[1.0], [1.0], [1.0]]], "dtype": "float64"}, {"__nd__": [3, 1], "dtype": "int64"}, {"__nd__": [[0.5], [0.5], [0.5], [0.5]], "dtype": "float64"}, {"__nd__": [[0.5], [0.5], [0.5], [0.5]], "dtype": "float64"}, {"__nd__": [0.5, 0.5, 0.5, 0.5], "dtype": "float64"}, {"__nd__": [0.5, 0.5, 0.5, 0.5], "dtype": "float64"}, {"__nd__": [[-0.5], [-0.5], [-0.5], [-0.5]], "dtype": "float64"}, {"__nd__": [[-0.5], [-0.5], [-0.5], [-0.5]], "dtype": "float64"}, {"__nd__": [-0.5, -0.5, -0.5, -0.5], "dtype": "float64"}, {"__nd__": [-0.5, -0.5, -0.5, -0.5], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.5143859225346401, -0.03715067452310716], [0.7612810361872527, -0.03593879277619472], [0.8406515508597837, -0.029851680305425975]], [[0.5143859225346401, -0.029851680305425975], [0.0, 0.0], [0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[0.8406515508597837], [0.5143859225346401]], [[-0.03715067452310716], [-0.029851680305425975]]], "dtype": "float64"}, {"__nd__": [[[2.0731060395815852], [0.740026109351293]], [[-0.203444429195402], [-0.16512214429357344]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.20007064978275851, -0.09887990363746195], [0.4747207458982391, 1.7586005225833379], [0.5328708987333687, -0.11857030102266883], [0.6695577234931344, 0.11588640746924374], [-1.6537599883688663, -1.8111469829351374]], [[-0.5333590032294453, 1.9909196988576745], [-0.2809960162428199, 0.6554372397249413], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], [[-2.0711917249528926, 0.16008423462141808], [-1.5632547813990356, 0.32948806583472684], [-0.19052748435405345, -0.4476879275817809], [0.5675824183810106, -1.7889043997707288], [0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [5, 2, 4], "dtype": "int64"}, {"__nd__": [[-0.027199003235610322, 0.37067876710764047], [-0.34239676694208754, -0.4032303639033179], [-0.5530775770652967, 0.5703307163437323], [0.3894891534238124, 0.43314532474921585], [-0.32249001110512515, -0.4959586015929635], [0.5374709412113176, 0.389005668547046], [0.27935492737214945, 0.05416432167271285], [0.17807911050156228, 0.3067670529362883], [0.04602934387404747, 0.23964823955494363], [-0.3531280981100413, 0.06949487105464526], [0.2383864300778216, -0.38937925121612493], [-0.24343641235735003, -0.5364203479886027]], "dtype": "float64"}, {"__nd__": [[0.3818146445110857, -0.3054763456710991, -0.045035609501849684], [0.26629675483078463, -0.5025701359528096, -0.48343883341753835], [0.20338879397228407, 0.2595591502795646, 0.07564596304504627], [-0.5429554713321556, -0.016363712052465982, 0.555258387171512], [0.061305935922874664, -0.4699931266599226, 0.011232530230423232], [0.03472225359185065, 0.2268748830328345, 0.1306707221119606], [0.11611126339004392, 0.13523122109484578, -0.26669259082233265], [-0.17891954108669558, -0.2069458675177544, -0.27834575591532706], [0.09253745968750346, 0.49634092668633883, 0.05298782380628229], [-0.26149280756450427, 0.17953461671431362, 0.0717759813373729], [-0.5716472254297441, -0.33287489957926103, -0.44824975357042196], [0.35181122412370897, -0.19557067232205935, 0.4768121940344312]], "dtype": "float64"}, {"__nd__": [-0.2841669475712513, -0.00016011650329590422, -0.03150375157859595, 0.19325606271010506, -0.27715533816931587, -0.28561703141759004, 0.3384400964433478, -0.3462644845463312, 0.2856216427229143, -0.27789811518953494, -0.5766089996083942, -0.14915896687006272], "dtype": "float64"}, {"__nd__": [-0.2941785003530197, -0.0644567196294577, -0.3260187182592319, 0.08963940722729243, 0.04648054429446145, 0.35233327985499696, 0.23396401770804198, 0.2039261846114317, 0.1485089367853343, -0.35049028064238913, 0.49335641392123136, 0.07049682868453026], "dtype": "float64"}, {"__nd__": [[-0.24364002896903852, 0.33674187017468327], [0.10341935787260512, 0.14167275214505046], [-0.008454733207649903, -0.22820005655248618], [0.3195716415247596, 0.37530509158863307], [0.5418966264829823, 0.22706790980240077], [-0.34664158257686917, 0.5124394893676869], [-0.016957665027458674, -0.33776095693583424], [0.061179679779815456, 0.24088285730147801], [0.3935196782065399, -0.24656206245890083], [-0.10602405799223344, -0.09388124648729043], [0.07698922828260069, -0.1230506622616246], [-0.24705074827142157, -0.16988635139686903]], "dtype": "float64"}, {"__nd__": [[-0.07890215708097968, 0.5724509291187335, 0.008593430292368406], [0.07316281685628283, 0.2071426502042108, -0.1695731964951055], [0.24142492733199336, -0.1610453033002775, -0.35417376571670905], [-0.3698128970039006, -0.34911715133606525, -0.4112488539632971], [0.004383057285081117, -0.09937341082935443, 0.03509225752252625], [-0.3399162963602795, -0.1398711939188549, 0.20148556428933595], [-0.135457687926833, 0.2881357250720061, 0.4879916870288803], [-0.41060965475478345, -0.5771384354949622, 0.002681559431625914], [-0.15522689212235496, 0.07770512180165154, -0.1224012148879647], [0.35965434241322464, -0.12664866643816647, -0.16067810019816112], [-0.02414366430432115, -0.4097084643223174, -0.1907196429016256], [-0.27697009356263474, 0.000349351671848197, -0.4261675829050103]], "dtype": "float64"}, {"__nd__": [-0.49884385094797606, 0.22275694699635942, 0.153495322308685, -0.1633341885958489, 0.03878006871372672, -0.1788506274396776, 0.21578169230188213, 0.5021419038290501, 0.5618526805207501, -0.4030563433820068, -0.4572857404532497, -0.25061839500963784], "dtype": "float64"}, {"__nd__": [0.46122499693464303, 0.48073186178032806, 0.48205134441613484, 0.569071499395617, -0.3607533068923106, 0.424718935484567, -0.23323908039495395, 0.16544150763096274, 0.08842767849092044, -0.49521701332681894, 0.487460109419329, 0.2696422564625607], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.06314481748930988, -0.032463724338228514, 0.07048142770996867, 0.011279657589844179, 0.28174864524513243, 0.3125543623164829], [0.15225836834059786, 0.03472062113601959, 0.12289066791366877, -0.027976701713775618, 0.29572675425661715, 0.2315044357454251], [0.14158362596074542, -0.011129264818434018, 0.2088620058718347, 0.04907886613905928, 0.24249584892589301, 0.2841306982177417], [0.15072587024894682, -0.016882378967650415, 0.19423813576471727, 0.04503549704284217, 0.20863329461866995, 0.23133975896612127], [0.060064595228347825, -0.2884278085283437, 0.04700176816746222, 0.08694564505775444, 0.03902373011777583, 0.20379312403829863]], [[0.10981529802673488, 0.032366496410502536, 0.12871581393421722, -0.10710251950594259, 0.26369683101493185, 0.07018214522061876], [0.13158522406263123, 0.00847949926617657, 0.23024097895949353, -0.0363867584872195, 0.20596832054577055, 0.10878508155750138], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0005058449616261634, -0.09323429475340837, 0.14556714793884817, 0.04568875705596921, 0.14580686409058466, 0.1203898805557747], [0.0201286139129075, -0.11429746244740505, 0.18174994037822215, 0.04760984102903449, 0.16469715409901478, 0.23912341448056118], [0.05495399599324426, -0.17058772161946284, 0.14321711569897008, 0.06470010362681462, 0.16390233737411353, 0.30468283108297], [0.05137478386319886, -0.3418519259474605, 0.06905266754413153, 0.05066924459652699, 0.09394051141475114, 0.3079127649146918], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[0.060064595228347825, -0.2884278085283437, 0.04700176816746222], [0.13158522406263123, 0.00847949926617657, 0.23024097895949353], [0.05137478386319886, -0.3418519259474605, 0.06905266754413153]], [[0.011279657589844179, 0.28174864524513243, 0.3125543623164829], [-0.10710251950594259, 0.26369683101493185, 0.07018214522061876], [0.04568875705596921, 0.14580686409058466, 0.1203898805557747]]], "dtype": "float64"}, {"__nd__": [[[0.13413604516631886, -0.6328687152602052, 0.058154334544814223], [0.3631012430264453, 0.02298646274109723, 0.5919504861322842], [0.19165256454670046, -0.5626720646212564, 0.09848074929492757]], [[0.04180979601041865, 0.6858093081284365, 0.7919426017563576], [-0.4527497645742009, 0.7610751295169318, 0.1592879331213892], [0.14200097186676114, 0.34903265500554725, 0.20484234058389167]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-2.129777118899729, 0.9952937261143133, -0.017934504651792728], [7.0, 7.0, 7.0], [7.0, 7.0, 7.0], [7.0, 7.0, 7.0]], [[-1.8243552152359248, 0.5052261393952611, -0.686750146781543], [0.8424969512705861, 0.3066769180362716, 1.506673709908708], [1.1795053000384128, 1.110042858574655, -2.1799671507692104], [1.3191183726548221, 0.6965349691175896, -0.035158413610980696]]], "dtype": "float64"}, {"__nd__": [1, 4], "dtype": "int64"}, {"__nd__": [[0.32690650419136547, -0.061798425016267956, -0.49729767555062176], [-0.23523389147484708, -0.027475644580927973, 0.23606473629580477], [0.13824077623919295, -0.2126315237634246, 0.35985497446994896], [-0.489105996183323, -0.21854515269813388, 0.4160653800523162], [0.21211484461940266, -0.011292445794433451, -0.18934971298351166], [-0.2390905811575789, -0.12352831898079365, -0.5356170398349464], [-0.5018572104570197, -0.2440097366523159, -0.3122001822104728], [0.45357596964761804, 0.01785381705621134, -0.5936333064518153]], "dtype": "float64"}, {"__nd__": [[0.06602487105295805, -0.5100176917648542], [-0.16406112164465309, 0.21127309585683515], [-0.24865126010632307, -0.047023730247284005], [-0.6273724484248316, 0.45156494146256243], [0.5605147975902756, 0.0522111935143959], [-0.28327493450404456, 0.6708383180867286], [-0.2620334453576204, -0.3916348940320924], [0.17756743603154754, -0.5415246970962654]], "dtype": "float64"}, {"__nd__": [-0.2535887804732034, 0.5788016307350673, 0.18646526217004267, 0.4635669826474053, 0.5886656959741752, 0.6059629802691768, -0.05037635527888954, -0.42199091850287196], "dtype": "float64"}, {"__nd__": [0.025611330949981064, -0.2008531265404808, 0.3083434039835292, 0.2076132929521754, 0.22860767013259634, -0.3842917692837942, 0.5344324423844908, 0.035446086517659736], "dtype": "float64"}, {"__nd__": [[-0.5199157409858144, 0.6031425735621152, -0.16560122813373498], [0.3781924876161619, -0.34944473611530313, 0.6857483745731534], [0.40436802079853923, 0.47801979217270496, 0.16440104632498503], [-0.05031925268970459, -0.27755038521173353, -0.3789677739811711], [0.22936212492269487, -0.2910322764855471, -0.4002427730571436], [-0.1252871190091556, 0.5558879252805837, 0.21448726586878528], [-0.41772804737182495, -0.2956569384297852, 0.043010647983374874], [-0.37924853319126856, 0.3326705866412193, -0.6468569241288079]], "dtype": "float64"}, {"__nd__": [[0.6692076171863974, 0.34460017566530926], [0.09536441813384584, -0.04070154340210371], [-0.13479308208566979, 0.5758379975604699], [0.46300041256531305, 0.18107010157204143], [0.5208131915004899, -0.4047395632708019], [0.24766358948847622, -0.3451085833621529], [0.6010326206019783, -0.06781096772375358], [0.6368708580929942, 0.04148383422284612]], "dtype": "float64"}, {"__nd__": [0.496605489247099, -0.5071868754164648, 0.00876675288053962, -0.6723174097727015, 0.12207532543315956, -0.4237753485392896, 0.4654599875707811, 0.2908175943361798], "dtype": "float64"}, {"__nd__": [-0.3170226465529245, 0.529511743575359, -0.5892984202618785, 0.24381462201891693, 0.6930491793863001, -0.26537018190973527, -0.4013736944957823, -0.2577022545676814], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.07379092258437253, 0.07705300669998576, 0.02524692283186579, 0.023390569247402924], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[0.1521419184798217, 0.1402812199063589, 0.3845164373382791, -0.120640976727024], [0.12014783513339256, -0.020805603388901923, 0.24503610901655495, -0.05282340925221986], [0.3900931734891219, 0.2090861826558346, 0.20730689609605515, -0.18834335120094045], [0.27722109067993284, 0.025341642770241708, 0.11215602091028591, -0.1090243672937931]]], "dtype": "float64"}, {"__nd__": [[[0.07379092258437253, 0.07705300669998576], [0.27722109067993284, 0.025341642770241708]], [[0.02524692283186579, 0.023390569247402924], [0.3845164373382791, -0.120640976727024]]], "dtype": "float64"}, {"__nd__": [[[0.09386395037906392, 0.38459665877635196], [0.9593799619085003, 0.046098922206443405]], [[0.038331342582113466, 0.030562134521862048], [0.6293675878321366, -0.1488889496663447]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0022662611210493, -2.0601901094499446], [0.5601554800869581, 0.10771576611968221], [-0.6188606694782937, 0.9895199955337259], [0.23089612338617577, -0.9439333980726978], [-1.5028327684166465, -0.10286001198973631], [0.5767258542663207, -0.8376366961607066]]], "dtype": "float64"}, {"__nd__": [6], "dtype": "int64"}, {"__nd__": [[0.0693954521329243, -0.028232428099964513], [0.29244537868397547, -0.17355577402159772], [-0.28916082357539075, -0.20008806423265546], [-0.35115351484498725, 0.3215429607122674], [-0.3269064612536581, 0.44933278934471577], [-0.03756810383294529, -0.4736517696740514], [0.12079420148019349, 0.14560675920890853], [0.10040928350139788, 0.4736229855390234], [-0.13087577743803858, 0.05853522943433431], [0.49499892050990824, 0.01298308875565135], [-0.34465168430589055, -0.09112048633486225], [-0.1578836416487699, -0.3563074025870385], [-0.22245468636013732, -0.4352120095715085], [0.23560603541009284, -0.3583205641745545], [-0.34946996822906684, -0.3440551958855852], [-0.362200162466475, 0.08087861076859171]], "dtype": "float64"}, {"__nd__": [[-0.08762507024833333, -0.1555159247182173, 0.022288335911524237, -0.04730473683040837], [0.18593796261236395, 0.3241440304287635, 0.033779263651612434, -0.21567002292430393], [-0.42731379775309986, 0.19038857526611286, 0.47637638450801345, 0.4591445822435485], [-0.117745010407597, 0.2441490637725363, 0.10376396138232413, -0.008706458201697354], [0.32165878745785936, 0.13345714648118112, -0.34850875963994643, 0.18415526181394415], [-0.09855750607745373, -0.07304686353514989, 0.4467369968246456, 0.45852015920612477], [-0.032081443094746365, -0.34774244875085647, -0.16343552941457573, 0.21941341690239768], [0.405491839477896, 0.41059885210433544, 0.3737984572766362, 0.32445604777509585], [-0.06766497609345867, 0.1616822805057887, -0.29588331233391574, -0.3287645677030637], [-0.03163782461454956, -0.2147268766496392, 0.020978776341877836, 0.46733027854005305], [-0.4233153776709415, 0.08850832477585613, -0.18832951037693169, 0.08577583482421502], [-0.14219905183704207, 0.18344637219174964, -0.4802368484504681, 0.1461673739452085], [0.20442649896698162, -0.05821997111612909, -0.27548023803136346, -0.46683866231144777], [0.1231896941652525, -0.21128420782108537, -0.49538374525400264, 0.4196816536325725], [-0.36346416689647154, 0.12154899633191452, -0.07960877529175314, 0.06564965070211448], [0.40102289486499887, 0.4989609022128344, 0.43399073532570587, 0.41740215859108554]], "dtype": "float64"}, {"__nd__": [-0.19677906296658865, -0.23251576335501356, 0.05249942431472676, -0.12508236835992603, 0.4747231240250922, -0.041594458645215715, 0.36113407996607216, 0.44505407622960735, -0.34792521836758017, 0.21598822555914432, 0.008078729461499123, -0.47650043186960234, 0.2876382178631274, -0.13935259152591828, -0.02042553845823536, 0.4792159419018718], "dtype": "float64"}, {"__nd__": [-0.4499805743646431, -0.34353175672063796, 0.049161428057898915, -0.2823955591448475, -0.457666520234834, 0.4020866260787487, -0.17553464004124342, 0.42790391735417344, -0.36762131247500396, 0.37652219319015423, -0.21720079311684148, -0.12224436643175207, 0.06401483485719583, 0.2021130093783733, 0.14886088764507766, -0.0862408406885502], "dtype": "float64"}, {"__nd__": [[0.12877327092908086, -0.4403478597947904], [-0.28008435183667135, 0.4660019876648548], [-0.4069790583932009, 0.2140971928461859], [0.24882625245118684, 0.06722083933199796], [0.06718394685971263, -0.2708582571080055], [-0.22637078050925674, 0.46646886055006953], [0.05952168391309609, -0.09207142551127856], [0.4031202973763739, -0.3899260986179025], [-0.16805871725293053, -0.380804307415363], [0.41230807750491527, 0.48663414709053976], [0.15541644472841032, -0.4724449731209063], [-0.20494924991876484, 0.3215013118444773], [-0.09916093340488996, 0.2712205913992367], [0.11467097858773534, 0.2141559357150542], [0.31867898564908914, 0.005621093409235889], [0.4054833330278541, 0.428621791663238]], "dtype": "float64"}, {"__nd__": [[0.14349122099726064, -0.10541854328378708, -0.19848243341672955, 0.21652045225008387], [0.267948674956089, 0.01048535434056952, 0.4677696202190441, 0.32938971815133056], [-0.2698989959138599, 0.13879614657065698, -0.18801899841156222, -0.12756385055264585], [0.14978171496149817, 0.025395376988613272, 0.2047647335999515, 0.2715486879069048], [-0.279879054163681, 0.34504582056786404, -0.1991964342400504, -0.4838952655675789], [-0.36853706556905985, -0.20968703959702162, 0.36485677868213173, 0.01907798628742463], [-0.11742508561667264, 0.15992528029301767, -0.4771930779158746, -0.4813834960547638], [0.3270685600259501, -0.0023256590457928494, 0.41980139111036685, -0.2313258470069064], [0.13288271442813748, -0.14079314415664945, -0.3174511549550153, -0.4017891365423478], [0.28973383632327643, 0.08286329625444733, -0.44862113937322934, 0.13081299605001462], [0.06807851661735376, 0.07824740133950692, -0.4101498637948052, -0.15831970556870822], [-0.44934761729619377, 0.09904681508448743, 0.29574554519681406, -0.08673606898884734], [0.07223552133920874, -0.43028756603062035, 0.13784650389279074, -0.20387486888272666], [0.1081101482268998, 0.31946758568157285, 0.08816256411607803, 0.2503169137409941], [-0.23904125852233327, -0.45302164105265674, -0.03529091361719994, -0.2988993463873628], [-0.21732678145267958, -0.17543698869592406, -0.18959875412275407, -0.3638829976970692]], "dtype": "float64"}, {"__nd__": [0.17666722494624199, 0.07459406097247545, 0.12228795649409963, -0.10072682472091232, 0.32846617964323976, -0.33271135186215595, -0.46172126106986355, 0.2672626642742627, 0.437085195307927, -0.10543544291338491, -0.37004772619827686, 0.13083783350907363, 0.3074375122326828, -0.37301154737604936, -0.3403640045672811, 0.3688630277125836], "dtype": "float64"}, {"__nd__": [-0.2007856729766505, 0.40131968635607773, 0.4467050010013749, -0.3129727411883775, 0.06786922656466243, -0.49273590224044883, 0.12787323898355796, -0.027558706120043408, 0.4126531550976117, -0.37824138284295283, 0.31224194874131006, -0.35551918392829773, -0.06634520573327718, -0.08049781598964045, -0.4342704972792705, 0.11354743771244324], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[-0.20009820690941924, 0.285460246634553, -0.11939348521165254, -0.0020755895521336323, 0.35205199737051945, -0.07711208665702614, 0.10517055689209627, -0.2696055185466394], [-0.17555761331486255, 0.24610308651864485, -0.12010536821885685, -0.11108931267827639, 0.3685529573612928, -0.05256197814379384, -0.03871875649233402, -0.18904657435887678], [-0.18075074746156813, 0.10462538790228519, -0.06265038104255233, -0.3058117517780533, 0.4025563290894297, -0.13914325701534927, -0.08550692127078356, -0.08136004924440556], [-0.22492170203579212, 0.21765763689081585, -0.07839198021397067, -0.18358646545664356, 0.3824574146085796, -0.14549134501268496, 0.06184839904902308, -0.17346915668369403], [-0.21257483337357136, 0.06321482207323867, 0.13490956353658676, -0.2456329192108523, 0.35152004972245976, -0.19885182298932005, -0.02141454797255067, -0.050462662833434056], [-0.21431026920556187, 0.2102641914647272, -0.026643902127762523, -0.17105437816146285, 0.21697737157769556, -0.09748583231454175, 0.07600418157270322, -0.13225257170484356]]], "dtype": "float64"}, {"__nd__": [[[-0.21431026920556187, 0.2102641914647272, -0.026643902127762523, -0.17105437816146285]], [[0.35205199737051945, -0.07711208665702614, 0.10517055689209627, -0.2696055185466394]]], "dtype": "float64"}, {"__nd__": [[[-0.34259945487237514, 0.3857658167716741, -0.04695358250685401, -0.3529028916793268]], [[1.234954471568265, -0.25542749809802723, 0.2811694564849222, -0.5988832006096296]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-1.4914843455302147, -0.5395521877953232, -0.17212742724981328], [0.8846558072528748, -1.7502557864419688, -1.3373682236240059], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0]], [[-0.718514500407426, 0.24295469520605575, 0.03555668601428542], [1.1831303125275237, -0.6772313099334428, 0.10593467994685175], [-0.6933228924397153, -0.4173782582208482, 0.7533608079863787], [0.9010975400524432, -0.745475801698684, -1.4170341723221602], [2.159665881361447, -0.22585052731145863, -0.6069632580207271], [1.220465975480193, -1.716803036662996, 0.8857755355247728]], [[-0.15240443870361733, -0.21370289848739865, -0.4385588102702496], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0]], [[-0.4052769971319596, -1.0655180042841297, 0.09036691942703647], [-1.4857395952408197, 1.65501093340731, 1.05368878134255], [-0.3881331950309163, -0.7093279356179317, -1.1068055606389138], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0], [-3.0, -3.0, -3.0]]], "dtype": "float64"}, {"__nd__": [2, 6, 1, 3], "dtype": "int64"}, {"__nd__": [[-0.2941344956718945, 0.10474543647569703, 0.3169524235909851], [0.11089842185057401, 0.3094773985438952, 0.062162484162358665], [0.2247222802831035, 0.5294959317240224, -0.00155655141489129], [-0.45867436071689033, 0.05563751259506196, -0.12283839535424712], [-0.42957155301921884, 0.3665829607075908, 0.376100104401193], [0.48989785169245087, -0.49520438196456185, 0.3684636202650362], [-0.1700058971766797, 0.35056556594051974, 0.4073957549252569], [-0.2795715400265992, 0.23204324468263157, 0.24177786904727513], [0.4588471841449231, -0.15043694960404394, -0.3626391435644362], [0.030324744553789567, 0.20906839294722138, 0.3737944861627446], [-0.4874092766538252, -0.22262253624528655, 0.13982925459561676], [-0.36210981073258836, 0.26912614487723047, 0.2169227479540741]], "dtype": "float64"}, {"__nd__": [[-0.40265176071097797, -0.06215352103043512, -0.30508040459316454], [0.04181958923436249, -0.13523165039115442, 0.28416063708190165], [0.18500401274552553, -0.06349626606643834, 0.16737838951935344], [-0.2861104477553223, -0.5466750964658723, -0.4154347251210586], [0.3057951061978503, 0.18664240459828824, -0.24532641111885387], [0.14445094874315959, -0.5466946718483536, -0.10029265660994317], [0.5444648247515611, 0.048304341280887, -0.18604495282815134], [-0.08508243270623296, 0.32755135286559456, -0.23980778252971105], [0.5474133298060182, 0.30354725831538665, -0.31014785137751405], [-0.43716905212188645, 0.3510426636190304, 0.24557775455013484], [0.1487502409896202, 0.3655755100958088, -0.49948194496582377], [-0.15037038879170087, 0.40640888300038913, -0.04193573994146561]], "dtype": "float64"}, {"__nd__": [-0.4805355907429911, 0.32928549378568933, -0.24635770794520193, 0.3765663622982521, -0.4056279113428246, -0.41739042385768943, -0.23477833836790474, -0.3229016058744111, 0.30539470668428415, 0.5464437066116411, 0.44957259351276585, 0.4254643406212059], "dtype": "float64"}, {"__nd__": [0.28984471646837445, -0.5191390352833383, -0.1987814707732413, 0.14610461759532153, -0.3214479619857404, 0.4217153188717822, 0.2047459512882921, 0.14836783566846273, -0.3189734692784282, 0.06536683690219873, -0.5001798296120185, -0.5752631720991944], "dtype": "float64"}, {"__nd__": [[0.4948329608168051, 0.16819163774527046, -0.020297141195157874], [0.07484064902802723, 0.1252925169642216, -0.31202216444811604], [-0.4677298800170727, -0.3321435670521701, -0.3303602588191246], [-0.23830032600910944, 0.23087885031813696, 0.20184396757071385], [0.482333643356539, -0.4800941453923516, -0.4257516180506052], [-0.4581360322007298, 0.3363071286959356, 0.49577433517693753], [0.1403924644424396, -0.5180117959208955, 0.2903179769478492], [0.5261904903399034, 0.28969803837779223, -0.4476902424872012], [0.308734557646721, 0.4488242247420866, 0.5640778557413983], [-0.25724701070256367, 0.11503643329640555, 0.42131186170685647], [0.06463980256295598, -0.12117658912466674, -0.05743731083972331], [-0.2661352994979473, -0.2522860547021555, -0.2678425026069054]], "dtype": "float64"}, {"__nd__": [[-0.15519459360246615, 0.0915295910710073, 0.3120821143064003], [0.1215399361324805, 0.14133228338508577, -0.40017702797345606], [0.518111098990772, -0.3474449041181237, -0.44091330789740313], [0.25705980530828915, 0.058553875186814645, 0.4113556638402308], [-0.2880804994826813, 0.018208287577112618, 0.24028618283068773], [-0.4741695180049409, -0.21764834142555892, 0.005941121710351898], [0.13763525982915206, 0.5298418465713565, -0.02492374169143627], [0.4881986304967936, 0.3869256667148005, 0.027522686520613814], [0.13097150073371697, -0.08768072243145904, 0.3056170159487616], [-0.018025432795479964, -0.1587448768053026, 0.2104006255136731], [-0.2734353167916581, 0.22282310847360143, -0.5297056121611575], [-0.16944059888962223, -0.43904230544204237, 0.329765849870545]], "dtype": "float64"}, {"__nd__": [-0.22051137019931277, 0.4133680035763363, 0.0964379790018886, -0.32362643937661384, -0.10861173906692181, -0.3207056236745723, 0.5178965950989388, -0.1459606518627322, -0.09860824242832802, -0.3952753381358476, -0.23069922951920396, -0.11007010320715824], "dtype": "float64"}, {"__nd__": [0.18786638000431813, -0.3037023337219599, 0.39681076313440433, -0.2032021002963525, -0.1829250892618653, -0.15870974290689288, 0.542785242597873, 0.25849473502647036, 0.1461072174397264, 0.11710554313142804, -0.19991555063992472, 0.17481837340841727], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[-0.011376345173027802, 0.019090739841394068, -0.0708241514278337, 0.18649406633582483, -0.08698846698155466, -0.42649883590105087], [-0.10357320162477121, -0.11164233800116874, 0.02341503052116808, 0.10129980681093065, 0.159564289707017, -0.3641049000844001], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.06391113453122498, 0.02313693680328039, -0.07752448982908698, 0.2682157415976434, 0.026121794812312922, 0.005332788966024], [-0.050846447818637903, -0.08943742429702274, 0.035223107167168956, 0.21399900910662578, 0.13462433075557245, 0.024691411646447538], [0.04050671072920676, -0.03918216346609253, -0.05613960589064059, 0.28719115650403015, 0.016858297093166708, -0.10132963205075174], [-0.08186549927113572, -0.11025690611695602, 0.055425016172255134, 0.13659266758070315, 0.3167890038847733, -0.2328152556439568], [-0.1449143238973963, -0.08948118863725768, 0.11843201392305418, 0.16009314579360967, 0.198884188059288, 0.06357645922109327], [-0.20480160939717992, -0.13032724239184454, 0.14645840786524336, 0.20070743635745458, -0.026684805487159576, 0.04040852822699259]], [[-0.06369543564469704, -0.05956077106971363, 0.01674614826685057, 0.1343707099509541, 0.036728886254414415, -0.12656004539214813], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[-0.07980210846067326, -0.06048562266315559, -0.00776767119109387, 0.18878469913515342, -0.11963590419712256, -0.20350626094708166], [0.34428832399013654, 0.18581190843205897, -0.3209483345220661, 0.19530398483277706, -0.09047958306496533, -0.017421949087909592], [0.0810344001929674, -0.027242649407605263, -0.014159179601315859, 0.1035426175238207, 0.04822580772453652, -0.3521790136792963], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[-0.10357320162477121, -0.11164233800116874, 0.02341503052116808], [-0.20480160939717992, -0.13032724239184454, 0.14645840786524336], [-0.06369543564469704, -0.05956077106971363, 0.01674614826685057], [0.0810344001929674, -0.027242649407605263, -0.014159179601315859]], [[0.18649406633582483, -0.08698846698155466, -0.42649883590105087], [0.2682157415976434, 0.026121794812312922, 0.005332788966024], [0.1343707099509541, 0.036728886254414415, -0.12656004539214813], [0.18878469913515342, -0.11963590419712256, -0.20350626094708166]]], "dtype": "float64"}, {"__nd__": [[[-0.23874455875888015, -0.25899319855545516, 0.10299743260974242], [-0.31871969754346147, -0.3074842353149171, 0.5492662020543604], [-0.10685732213356607, -0.11909302032854328, 0.03820833386177391], [0.1747136551094762, -0.045152070143769596, -0.035361192858912344]], [[0.4230337295313219, -0.2014820697606668, -0.8714488931846284], [0.6270324098399364, 0.07152548110578193, 0.010168449224679686], [0.35947525746212927, 0.0911951345551286, -0.22631577300523237], [0.4617250281853263, -0.30690248490956745, -0.3512380540542462]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.5941490082123966, -1.8701215049132263, 0.037700833328394405, 0.4671269599854102], [0.03796728276966695, 0.7042022353521896, 0.11022150589678903, -2.284155945541785], [-0.7695511596347915, 2.1767018095162154, -0.4624363778094625, 0.5172215012049223]], [[0.9239541596433385, 0.8112342382436278, 0.3265319177544263, 1.4339609360926189], [0.23903057596285937, 1.5893966105719812, 0.41348358424358944, -0.607153534562892], [-0.3376276222604124, -0.25972807747604476, -1.030734554711556, -0.20357133750943376]]], "dtype": "float64"}, {"__nd__": [3, 3], "dtype": "int64"}, {"__nd__": [[-0.3221027740480106, 0.25858639075016693, -0.16598671124182188, -0.09067574222358356], [-0.16571086638352164, -0.4154274958213754, -0.013451318686002625, -0.436471491903945], [0.4208884428376716, -0.1234403042889438, 0.40577413538215024, 0.42202188256683126], [-0.26540603253844613, -0.28061455551399084, -0.3863480245040627, 0.18231699167757032], [-0.08035158647393634, -0.2273709971684772, 0.30376283875901056, 0.26298813304139934], [-0.39355505270635854, 0.4209533001271492, 0.05488571791070732, -0.4170268461853045], [0.3600151177069459, -0.09917133329393506, -0.18427976832462273, -0.22872977597082206], [-0.3776150920827273, -0.07088248073094613, -0.23799794426066623, 0.2581964412684541], [0.24964615804138476, -0.1317182152943476, -0.12658039011151273, -0.2572751386525238], [0.44659651677369105, -0.1953438287648422, 0.31896777297454576, -0.13857244165918556], [-0.03713665553765484, -0.16716017682314444, 0.32305539238594405, 0.02421237235631874], [-0.40597579798106265, 0.3964129903284403, -0.29993382828953546, -0.25123723196977416], [-0.06733699010073468, -0.20523279206690584, -0.14736110635818314, 0.18046307717722554], [-0.2507018447418135, 0.04818467511760455, -0.3449745798067271, 0.0910561608704428], [0.029102724030978133, -0.37565449393319283, -0.10294137853872248, -0.05879275416325058], [-0.3383533921276936, -0.16635198154393133, -0.13057447978321565, 0.01311287137452513], [-0.14869047926218645, 0.31518508084228725, 0.06712722383139146, 0.4143513572889215], [0.36349609874759076, 0.0768090234429378, -0.13298177794177715, -0.07376482156158894], [0.3499366955805456, -0.36280482858787183, -0.17631532399072283, -0.14184098272613682], [0.03303123796472774, 0.42339142444451927, 0.0005360236680655047, -0.06288793597082386]], "dtype": "float64"}, {"__nd__": [[-0.09835478961684498, -0.14252585166828757, -0.11000755890564769, -0.10634366208848685, 0.2514542925824178], [0.16163096058672574, -0.12640231560464865, -0.26286974422122605, 0.3500995625875939, 0.28891637815596993], [-0.2516033887416537, 0.13774080431528318, 0.2623279156702684, -0.12931160650372378, 0.03344954941974937], [-0.37733046955824445, 0.16152831979012466, -0.1941910291559189, -0.33485966788623045, 0.31687281344363155], [0.23037790942600583, -0.38839283967902094, -0.20592633605664398, -0.04283058202871518, 0.04873757642074497], [0.21095934102456035, -0.1291552359825317, 0.17439576701334208, -0.1983200223404593, -0.33085407337307643], [-0.3115565466395738, -0.15618936798343616, 0.06472480551341185, -0.06911966720469959, 0.24805034761130848], [-0.13856981983034433, 0.30617365531802626, -0.3618526255980927, -0.024191091244522175, -0.14855298930290214], [0.36736068255139326, -0.19802817651444551, -0.051482265017233936, -0.1954126271692217, 0.16302380086317836], [0.35826145139424503, 0.2168935758867971, -0.3918167187052293, 0.369828172636796, 0.40014116507345937], [-0.014052721348403107, 0.4164299760107755, -0.14957288561403234, 0.3418039410791114, 0.37351853340858704], [-0.42189226164337895, -0.35184712015584113, 0.06216657287926748, 0.06956465561831347, 0.1638856227781662], [-0.43896738226615295, 0.41391880629999506, -0.4124179817698496, -0.30796403036279435, 0.42524484339453444], [0.1008723757064536, -0.42363174755298044, 0.4056107715560486, -0.09716446466745851, -0.2623181270274645], [-0.1603672963292055, 0.35438729102436284, 0.09630792960623, 0.21018814751186743, 0.27335293060648663], [-0.017257906653568945, 0.1274913074297043, -0.07055065622655661, -0.2574874073897827, -0.37724725840048906], [0.28750526284518113, -0.0634929830962704, 0.401242426219722, -0.29562208145057484, 0.2426629608669519], [0.0991918243197856, -0.40919964904831835, 0.09624007866297912, 0.37329249454050945, -0.3742540027404157], [0.11567501794519627, 0.03862074494586033, -0.2879650182385266, -0.4453281985850706, 0.2796186729078207], [0.147126142805528, 0.09422505801664412, 0.05379460471759994, 0.24432870279378624, -0.2852720998199112]], "dtype": "float64"}, {"__nd__": [0.14879068590461358, 0.211489526998144, 0.24950542781810797, -0.40348006476833803, -0.24958904907088206, 0.17435593506201413, 0.09091412437636809, 0.33136635314085255, -0.3652656653236843, 0.29791325390140316, 0.268916939092798, 0.17224819176642928, -0.15096549934320572, 0.10663152241395664, -0.2046754520137816, 0.37361136110405296, 0.252354246594953, -0.1997023288844667, 0.30768595346605065, -0.15410365135138543], "dtype": "float64"}, {"__nd__": [-0.10620782733194419, -0.19979297050161304, 0.2986289908366156, -0.27142268577980494, 0.15677294014673712, -0.26412070294417306, 0.11795263749021509, 0.3360044390977408, -0.31563214627870356, 0.4226893075738491, 0.32965446931731923, -0.2357152134643245, -0.24219496542162464, -0.37416505801895317, -0.14123226062161232, 0.04767517341259581, -0.2926240192360441, 0.3468718518072358, -0.396834097111892, -0.08347187708965965], "dtype": "float64"}, {"__nd__": [[-0.13166060154414125, 0.015454910948239371, 0.4080520046215026, 0.19590903798791173], [-0.10304422386872894, -0.38177980815078605, -0.17565136623167218, 0.10837805943821226], [-0.2347244676051952, 0.15395073894226718, -0.43359004805557133, -0.0799210039390283], [-0.10736333073398795, 0.18352413058985662, 0.2478480576041688, 0.13289375305435464], [-0.1644806294368707, -0.010497274265955097, -0.31889975788904446, 0.2594260343852478], [-0.1961513068355354, 0.13642812230657284, 0.40407849033551335, -0.15493690146838765], [-0.15980244265680768, 0.2062692854016885, 0.09831315358390273, -0.18326767459562546], [0.21378015008996668, 0.11790895577964473, -0.2764370347318045, 0.31451547728521845], [-0.4033993633816163, 0.316208604963766, 0.4014374383677172, -0.43541869306897024], [-0.06195275821788183, 0.41041401864049093, -0.4313014845616663, -0.03579484459710481], [0.36495238303026634, -0.35346917228856173, 0.425223653577639, -0.27319118019748223], [0.32791852522485687, 0.06732518698580413, 0.4201401512129315, 0.12006979220099263], [-0.21020492916367983, -0.19861677416023304, 0.031178470440380527, 0.221243997034537], [-0.41164421883007346, 0.21806288265851442, 0.16911559508014343, -0.25943529117793324], [0.27434238232451086, 0.4306140878449172, 0.15795448818590904, -0.10317099319982218], [-0.2548036706086567, -0.28609612861333517, 0.05404577414868079, -0.08066524337854167], [0.3814756655798664, -0.19547275109727344, -0.29754651951383293, 0.027350176618816086], [-0.11670998220651636, -0.21045867020703132, 0.03482735421440908, -0.4117506227718438], [0.06598251539763907, -0.159480112273572, -0.3318952822799425, -0.3513636499481008], [0.374251843392274, 0.12523877266836686, 0.32836674827913986, -0.18952178708085665]], "dtype": "float64"}, {"__nd__": [[-0.0331557887268575, -0.08585105233207756, 0.18144579439656214, -0.2601132481658294, 0.013305709861958052], [0.03521853705826666, 0.39354583129312176, 0.4173984667240686, 0.044870153642038495, 0.4426279549830561], [0.26085683647454966, -0.17607860512702925, -0.2953932330730783, -0.3544735020576878, -0.37371702831563275], [-0.3453147875445843, -0.4341972419155174, 0.35646543632321215, -0.31412930980844, -0.08629704538991285], [-0.3868162320438639, -0.37039229293016, 0.04449637610969211, 0.09975253373408982, 0.3790183290898865], [-0.2587389658506543, 0.2999985209636068, 0.36372838284144904, 0.3721250021751732, -0.11761253383971701], [-0.1610934601763828, 0.3863537569829414, 0.17711462095486208, -0.39113867729395896, 0.12945402535798842], [-0.3702231360397919, 0.19187359580339225, -0.0011809314371505953, 0.3169232249626831, -0.05272523525525346], [-0.2445132765919002, 0.3839917787215505, -0.0129419203317315, -0.14386036358688128, -0.18985813548253094], [0.2741241300281239, -0.3907546188958177, -0.2428594393658426, 0.40090842442595587, -0.36586015464115895], [0.12330772214075289, -0.05428484276653141, -0.17587725478805227, -0.17431122246994965, 0.36912931919903147], [0.2833285202206197, 0.03752126496759275, -0.15116227660758766, -0.41396264466023824, -0.2902793801040517], [0.3888278259830475, 0.009605793832394665, -0.4027392681677228, 0.06956891232634761, -0.30734355547273934], [0.0391719532025449, -0.07795348602724206, -0.1405865224059575, 0.36758673874286496, -0.08040783056084055], [0.20321655506168418, -0.35596662400803053, 0.4419204476185867, 0.23126236994635327, -0.24560244479287038], [-0.000510549772018597, 0.432239852390489, -0.2788401539876448, -0.24891395631371746, -0.018031105584252513], [-0.08466715003983012, -0.22422859462156686, -0.21711836585855357, -0.08062519526523608, -0.10792511112132053], [0.28554063317984485, 0.3417071492904488, -0.15888645503555213, -0.05093409861004346, -0.34171282737869413], [0.16853300051738393, 0.06898894751018961, -0.17851972935499238, -0.19640396621573847, -0.2508891762639986], [0.4183481606310341, 0.16207381440860058, 0.0353285163543786, 0.026478014722163733, 0.21719961972501767]], "dtype": "float64"}, {"__nd__": [0.395653181072991, 0.16285072392445307, -0.3757474722723144, 0.05818179754239406, 0.11190107416293932, -0.03513660554183561, 0.18263192866646302, 0.23936308890044655, -0.2806060168485241, -0.01042630257477885, 0.050208138266919417, 0.13585336469726964, 0.10797590602382041, -0.07299014279127947, 0.3333935675920047, -0.34432403726238414, 0.42128689546088116, -0.1405156113456955, -0.33987670917859253, 0.035553933567783066], "dtype": "float64"}, {"__nd__": [0.01035483783255631, -0.11402143666435531, 0.12533539705268038, -0.1588299211637962, 0.21643614720180335, 0.3452279516263251, -0.34103842532928463, -0.3671609500451117, 0.2872136635003074, -0.40311154580901076, 0.18921823471312396, -0.09527664064255291, 0.038276332977871586, -0.42811711007449105, -0.24581548067135525, -0.2688067108327876, -0.37671110875163427, -0.2674456717006417, 0.11391926021454495, -0.040889874752960374], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.10240104019275609, -0.16480655312457343, -0.020164878718396055, -0.17154733782794826, 0.054781010488489136, 0.1681492701423441, 0.16385646691362438, -0.026425254790734848, -0.14154242474346007, -0.07828767528708136], [0.19060057920659287, 0.02491620272216228, -0.17564689371539174, -0.09990431339339614, -0.012666184583137513, -0.011803563466034941, -0.05714024572896038, -0.14389604923440416, 0.08174517136668182, 0.25512239538089443], [0.19733694045129474, 0.17238024209445724, -0.22803419530952396, 0.002579602874772015, -0.21212263781880664, -0.11480761859835696, -0.02623387833442166, -0.003934493893796553, 0.015524539046916518, 0.16451571357338848]], [[0.10919782527979267, -0.08542964540564363, -0.1864538733483575, -0.046353478551920374, -0.16511946109334277, 0.02036447972170295, 0.13076127067801419, -0.009601650648534686, -0.12903573409359834, 0.22949434792872445], [0.17974359386336047, 0.0511153492945832, -0.327133503601723, -0.04074070407120041, -0.30841407178407493, 0.000978207851571674, -0.023948362122156116, -0.04218451437802081, -0.06850055683121968, 0.1817928053125232], [0.2187944438361281, 0.05876447313828495, -0.2864214556094114, -0.0007654924905719775, -0.1789733828053936, -0.03249144107431331, -0.16008058441916329, 0.04604776943385658, -0.10809469980637365, -0.06386570991794922]]], "dtype": "float64"}, {"__nd__": [[[0.19733694045129474, 0.17238024209445724, -0.22803419530952396, 0.002579602874772015, -0.21212263781880664], [0.2187944438361281, 0.05876447313828495, -0.2864214556094114, -0.0007654924905719775, -0.1789733828053936]], [[0.1681492701423441, 0.16385646691362438, -0.026425254790734848, -0.14154242474346007, -0.07828767528708136], [0.02036447972170295, 0.13076127067801419, -0.009601650648534686, -0.12903573409359834, 0.22949434792872445]]], "dtype": "float64"}, {"__nd__": [[[0.33808732247702306, 0.2435730206167949, -0.48742932723721755, 0.009772750729566793, -0.3378463646073557], [0.32166547622311714, 0.14493956619877152, -0.5705581874050348, -0.0014267240125133973, -0.4354606971089484]], [[0.48930741576620745, 0.22608541356523879, -0.06936259566888235, -0.30048710340437085, -0.13601376709085894], [0.08576245172617343, 0.2446536705997676, -0.045772050535456466, -0.49346376319860985, 0.4210083585178638]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.04819955778413464, -0.4534225910151214], [-0.4101971518366532, -1.2410818504592633], [0.07341128034602527, -0.25551048228833256], [-0.3400139877491439, 0.48760613225417054], [0.3776018691910311, 0.698848226929813], [-1.7288785097764012, -0.2790143408406819], [-0.514711936864358, 0.37132645013436116]], [[-1.5325090013170557, -0.5637478173073662], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0]], [[0.33198243360025825, 0.7157181934318377], [1.5932237517929833, 0.7913134253534949], [0.9371193727923091, 0.5486856239451072], [-1.8201478843888506, -0.18501118001898564], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0]], [[2.2598411730598196, 0.19086293583934313], [-1.0402156238739633, 0.7919762879654201], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0], [100.0, 100.0]], [[-0.7941664150582687, 0.6171798552995965], [0.13093578347634885, 0.20531563439577605], [1.739844901976931, 0.1098589595675823], [2.174041106572087, 0.4717011010480132], [-0.008792699062933946, -0.003439229844280369], [100.0, 100.0], [100.0, 100.0]]], "dtype": "float64"}, {"__nd__": [7, 1, 4, 2, 5], "dtype": "int64"}, {"__nd__": [[0.07765977145781265, -0.20849100902501094], [0.13844283422915232, 0.23349870315497712], [-0.16215611249751652, 0.48896123137732134], [0.3872315483389229, 0.14792057385260107], [-0.06464131242308209, -0.5923529212050586], [0.4134442647935195, -0.05867322554539485], [-0.6701339961525196, -0.06826373949949997], [-0.543871047834919, -0.6387809905682749]], "dtype": "float64"}, {"__nd__": [[-0.39919071519231963, -0.1904227221556815], [0.27463883024984687, 0.2858720973137558], [-0.5488871467440862, 0.31591669132595734], [-0.6668612529851005, -0.070782271294597], [-0.5320619037937182, 0.4264660866884741], [0.44665028838416265, 0.2285627642102235], [0.1922467381726271, 0.28670739257635797], [0.4018020736091947, 0.25754212602465587]], "dtype": "float64"}, {"__nd__": [0.6500621395889358, 0.6461529938339141, -0.1549533662141193, 0.5833951141561559, 0.6159890887925976, 0.6758899733247147, -0.619411260506448, 0.7015922512218511], "dtype": "float64"}, {"__nd__": [-0.2122410357852938, 0.6116964918423804, -0.7036031339865594, 0.0367759372883617, -0.6322089843477239, -0.07831380288295088, 0.04337936338897523, -0.3876459241214936], "dtype": "float64"}, {"__nd__": [[-0.5427054627959608, -0.6909866858443232], [0.41946845000886823, 0.5670923751847042], [0.6760097833433482, -0.05601604406764739], [0.1878741955846947, -0.6161153917017929], [-0.0830224425847431, -0.6439557375196868], [-0.12407533600182519, 0.3242223445463527], [-0.5475579425136385, 0.1007879254222936], [0.05899872939455468, 0.5974505302316219]], "dtype": "float64"}, {"__nd__": [[-0.3034173005369229, 0.510551169826907], [0.32158668916080624, -0.21552016380643463], [0.21989932097573, -0.5965846362252], [0.26185647672778384, -0.4013709266899053], [-0.649777622343151, 0.46800628623311646], [-0.26816785931530335, 0.30762431424932307], [-0.705653949294482, -0.5885537827430831], [0.4845216425977297, 0.45989970173017874]], "dtype": "float64"}, {"__nd__": [0.34170108356602336, 0.4383157360817302, -0.6701584008563658, -0.3094883577899115, -0.651463533654539, -0.19812402067997098, -0.6886755731728048, -0.6196855069801235], "dtype": "float64"}, {"__nd__": [0.3521605947200138, 0.1427478836967131, -0.5893996703185862, 0.4172506474275491, -0.5026594375666267, 0.08532307199686595, -0.6758497776219152, -0.2967972606868822], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.05844553065150493, 0.2533036079020566, -0.10930830396666336, -0.05354198011356786], [0.20384501618059936, 0.43921858578268125, -0.07422239608164838, -0.02678980056484314], [0.09536753110818777, 0.4738417514833732, -0.11751964885715435, -0.0038990843701954005], [0.01083273291562794, 0.44392364097246456, -0.14897857424533228, 0.04468273596186991], [-0.04409083837613953, 0.3780129658816205, -0.11269727864053508, 0.04055135303283838], [0.12623992397174688, 0.42400181647117235, -0.21781635447710612, 0.00973943096275964], [0.010466141723573116, 0.3866929527235436, -0.13734306679922173, 0.014887938015629737]], [[0.14567407552019185, -0.0017263276091883805, -0.1674138614500669, -0.008813659415251881], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[-0.07328610791890544, 0.1899102552157215, -0.1060541307328203, 0.02336885882975978], [-0.05342331743512872, 0.20929523587131119, -0.05751993574902645, -0.006274765326547798], [-0.06431096131130397, 0.3347909745627685, -0.08726458876251054, 0.006224654186812222], [0.0714372403530086, 0.4476541331642804, -0.21975396244355874, 0.005527608374514767], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[-0.018502643925690374, 0.16748852666021513, -0.04348054944448164, -0.03652378578176211], [-0.1300385851512396, 0.3056292243479038, -0.18007155311893913, 0.06399269465204807], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], [[-0.08480073031357101, 0.10428756532521256, -0.1940230523189719, 0.010111507884273422], [-0.03171518600380457, 0.2676030351421022, -0.1260134766710976, -0.06991308067273777], [-0.010150583509981517, 0.28337048811351573, -0.05529110975973555, -0.10780077839262409], [-0.024158051977603717, 0.23450560498121248, -0.04275679093170294, -0.07652274417445348], [-0.0018245605548705367, 0.5427477094606099, -0.1017398806155054, -0.02046839987328799], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]], "dtype": "float64"}, {"__nd__": [[[0.010466141723573116, 0.3866929527235436], [0.14567407552019185, -0.0017263276091883805], [0.0714372403530086, 0.4476541331642804], [-0.1300385851512396, 0.3056292243479038], [-0.0018245605548705367, 0.5427477094606099]], [[-0.10930830396666336, -0.05354198011356786], [-0.1674138614500669, -0.008813659415251881], [-0.1060541307328203, 0.02336885882975978], [-0.04348054944448164, -0.03652378578176211], [-0.1940230523189719, 0.010111507884273422]]], "dtype": "float64"}, {"__nd__": [[[0.02216014328760582, 0.7207454144811554], [0.2394413655519798, -0.002108631127649169], [0.10591646337166463, 0.6173913612972477], [-0.2515366425305947, 0.5607900663345905], [-0.00485487504193724, 1.5700752085859857]], [[-0.5718759458576361, -0.2429565072974841], [-0.5063316107279583, -0.04262432236823078], [-0.6171955214102433, 0.06197858239991764], [-0.6469796179564034, -0.11255509937432423], [-0.6925640796237313, 0.0302361365341305]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(bilstm_padded, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")